# PneumoScan -- Ensemble Model

This notebook builds an **ensemble** of the top-performing models trained in earlier notebooks.  
We compare **soft voting** (simple probability averaging) with **weighted voting** (AUC-weighted averaging) and evaluate both strategies on the held-out test set.

---

## Why Ensemble Models?

Individual deep-learning models each have their own inductive biases:

| Architecture | Strength |
|---|---|
| ResNet-50 | Residual connections allow very deep feature hierarchies |
| EfficientNet-B0 | Balanced depth / width / resolution scaling |
| DenseNet-121 | Dense connections encourage feature reuse |
| MobileNetV2 | Lightweight inverted residuals -- fast inference |
| Custom CNN | Task-specific architecture, fewer parameters |

By combining predictions from multiple architectures we can:
1. **Reduce variance** -- errors from one model are averaged out by others.
2. **Improve calibration** -- averaged probabilities tend to be better calibrated.
3. **Boost recall on minority classes** -- different models may excel on different classes.

## 1 -- Setup

In [ ]:
# === PneumoScan Setup ===
import os, sys, subprocess

# Detect environment and configure paths
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_DIR = '/content/PneumoScan'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/muhammadrakib2299/PneumoScan.git', REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    
    SRC_DIR = os.path.join(REPO_DIR, 'src')
    IN_COLAB = True
    print(f"Running on Google Colab | src: {SRC_DIR}")
except ImportError:
    SRC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'src')
    IN_COLAB = False
    print(f"Running locally | src: {SRC_DIR}")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# === Imports ===
import config
import data_loader
import ensemble
import evaluate
import utils

import numpy as np
import matplotlib.pyplot as plt

# Configure paths for Colab (dataset on Google Drive)
if IN_COLAB:
    config.setup_colab()

config.ensure_dirs()

print(f"Project root: {config.BASE_DIR}")
print(f"Dataset dir:  {config.RAW_DATA_DIR}")
print(f"Models dir:   {config.MODELS_DIR}")

## 2 -- Load All Trained Models

In [ ]:
models = load_models()

print(f"\nLoaded {len(models)} model(s): {list(models.keys())}")

## 3 -- Load Validation & Test Data

In [ ]:
# Validation split (from training directory)
_, val_ds = load_train_val_split(augment=False)

# Test set
test_ds = load_test_dataset()

print("Datasets loaded.")

## 4 -- Rank Models by AUC-ROC on Validation Set

We rank every loaded model on the **validation** set so the test set remains completely unseen during ensemble construction.

In [ ]:
rankings = rank_models_by_metric(models, val_ds, metric="auc_roc")

print("\n--- Model Rankings (AUC-ROC, validation set) ---")
for rank, (name, score) in enumerate(rankings, 1):
    print(f"  #{rank}  {name:20s}  AUC-ROC = {score:.4f}")

## 5 -- Build Ensemble (Top-3 Models)

`build_ensemble` selects the top-K models by AUC-ROC and computes per-model weights proportional to their validation AUC.  
The ensemble configuration is also saved to `models/saved/ensemble_config.json` for later use.

In [ ]:
top_models, weights, ensemble_cfg = build_ensemble(models, val_ds, top_k=3)

print(f"\nEnsemble members : {list(top_models.keys())}")
print(f"Weights          : {weights}")

## 6 -- Evaluate Ensemble on Test Set

In [ ]:
results = evaluate_ensemble(top_models, weights, test_ds)

## 7 -- Soft Voting vs Weighted Voting

### How They Work

| Strategy | Formula | When to prefer |
|---|---|---|
| **Soft voting** | $\hat{p}_c = \frac{1}{K}\sum_{k=1}^{K} p_{c}^{(k)}$ | Models have similar performance |
| **Weighted voting** | $\hat{p}_c = \sum_{k=1}^{K} w_k \, p_{c}^{(k)}, \; \sum w_k = 1$ | One model clearly outperforms others |

Soft voting treats every model equally, while weighted voting assigns more influence to stronger models (here, weights are the normalised AUC-ROC scores from the validation set).

In [ ]:
# Build a comparison table
comparison_rows = []
for strategy in ["soft_voting", "weighted_voting"]:
    r = results[strategy]
    comparison_rows.append({
        "Strategy": strategy.replace("_", " ").title(),
        "Accuracy": f"{r['accuracy']:.4f}",
        "F1 (Macro)": f"{r['f1_macro']:.4f}",
        "AUC-ROC": f"{r['auc_roc']:.4f}",
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

In [ ]:
# Quick bar-chart comparison
metrics_to_plot = ["accuracy", "f1_macro", "auc_roc"]
strategies = ["soft_voting", "weighted_voting"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = np.arange(len(strategies))

for ax, metric in zip(axes, metrics_to_plot):
    vals = [results[s][metric] for s in strategies]
    bars = ax.bar(x, vals, color=["#2196F3", "#FF9800"], edgecolor="black", width=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace("_", " ").title() for s in strategies])
    ax.set_title(metric.replace("_", " ").title(), fontsize=13, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", fontweight="bold")

fig.suptitle("Soft Voting vs Weighted Voting", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 8 -- Confusion Matrices for the Ensemble

We plot confusion matrices for both voting strategies to see where each strategy makes errors.

In [ ]:
# Collect true labels from test set
y_true_list = []
for _, labels in test_ds:
    y_true_list.append(labels.numpy())
y_true_onehot = np.concatenate(y_true_list, axis=0)
y_true = np.argmax(y_true_onehot, axis=1)

print(f"Test samples: {len(y_true)}")

In [ ]:
# --- Soft Voting confusion matrix ---
soft_preds = np.argmax(results["soft_voting"]["predictions"], axis=1)
plot_confusion_matrix(
    y_true, soft_preds, "Ensemble (Soft Voting)",
    save_path=os.path.join(config.CONFUSION_MATRICES_DIR, "ensemble_soft_voting_cm.png"),
)
plot_confusion_matrix_normalized(
    y_true, soft_preds, "Ensemble (Soft Voting)",
    save_path=os.path.join(config.CONFUSION_MATRICES_DIR, "ensemble_soft_voting_cm_norm.png"),
)

In [ ]:
# --- Weighted Voting confusion matrix ---
weighted_preds = np.argmax(results["weighted_voting"]["predictions"], axis=1)
plot_confusion_matrix(
    y_true, weighted_preds, "Ensemble (Weighted Voting)",
    save_path=os.path.join(config.CONFUSION_MATRICES_DIR, "ensemble_weighted_voting_cm.png"),
)
plot_confusion_matrix_normalized(
    y_true, weighted_preds, "Ensemble (Weighted Voting)",
    save_path=os.path.join(config.CONFUSION_MATRICES_DIR, "ensemble_weighted_voting_cm_norm.png"),
)

## 9 -- Summary & Discussion

### Key Takeaways

1. **Ensemble > any single model** -- even a simple soft-voting average of top-3 architectures typically outperforms the best individual model on AUC-ROC and macro-F1, because the models make *different* mistakes.  
2. **Weighted voting** can provide a small extra boost when one model is clearly stronger, because it down-weights the weaker contributors.  
3. **Diminishing returns** -- adding more than 3 models often adds compute cost without meaningful metric improvement; the marginal benefit of the 4th and 5th models is usually negligible.  
4. **Clinical perspective** -- in pneumonia screening, maximising **recall** (sensitivity) for BACTERIA and VIRUS is critical. The ensemble helps here because averaging probabilities tends to push borderline cases above the decision threshold.

### When *Not* to Ensemble

- **Latency-sensitive deployment**: ensembles multiply inference time by the number of models. For real-time applications, prefer a single model or knowledge distillation.  
- **All models are near-identical**: if the architectures or training runs produce highly correlated predictions, the ensemble adds little diversity.

---

**Next notebook**: `08_evaluation_comparison.ipynb` -- comprehensive evaluation and model comparison.